## EDA – Analyse exploratoire des datasets contacts (CremeCRM / ISEN)

### Contexte

- Projet de création d'un CRM sur CremeCRM.
- Réception d'un premier fichier de données contenant des informations de contacts (entités, individus, adresses e-mail),
- ces données constituent une première extraction et seront enrichiers au fil du temps.

Avant toute intégration dans le CRM, une **EDA (analyse exploratoire des données)** est réalisée afin de :
- vérifier la structure des données,
- identifier d’éventuels problèmes de qualité,
- préparer les règles nécessaires à un futur ETL.

### Objectifs de ce notebook

Ce notebook a pour objectifs de :
- comprendre la structure et le contenu des données reçues,
- détecter les incohérences, valeurs manquantes ou formats hétérogènes,
- formuler des constats utiles pour préparer un ETL fiable.

### Périmètre

- Ce notebook se concentre uniquement sur l’EDA.
- Aucune transformation définitive ni import dans le CRM n’est réalisé à ce stade.
- Les données sources brutes sont stockées dans `data/raw/` et ne sont pas versionnées.
- Le notebook est versionné afin de conserver la traçabilité de l’analyse.


## 1. Description du fichier source

Informations générales (infos visibles sans code)

- Nom du fichier :  
  `CONTACTS DIRECTEUR-DRH-SERVICE RH-TUTEURS 15-01-2026.xlsx`
- Format : Excel
- Date de réception : janvier 2026
- Nombre de feuilles : 3

Description des feuilles :

Feuille 1 — CONTACTS DIRECTEUR-DRH-SERVICE

Cette feuille contient des informations relatives :
- aux entités (entreprises, organisations),
- aux individus associés (directeurs, DRH, responsables RH, tuteurs),
- aux adresses postales,
- aux coordonnées de contact (notamment e-mail, lorsqu’il est renseigné).

Volumétrie observée/visible :
- lignes : 6 799
- colonnes : 14

Premières observations générales :
- présence de champs vides,
- emails manquants,
- hétérogénéité des formats (espaces, majuscules, accents).

Les colonnes peuvent être regroupées par thématique :

**Identité entité**
- `Code.Entité` : identifiant / nom de l'entreprise
- `Libellé.Entité` : nom de l'entreprise
- `Code.Type d'entité` : type de structure
- `Assujetti.Entité` : assujettissement à la taxe d'apprentissage (vrai ou vide)

**Adresse**
- `Rue (ligne 1).Adresse` : adresse principale
- `Rue (ligne 2).Adresse` à `Rue (ligne 4).Adresse` : compléments d’adresse
- `Code postal.Ville` : code postal
- `Nom.Ville` : nom de la ville

**Contact**
- `Libellé.Titre` : civilité (Madame / Monsieur)
- `Nom.Individu` : nom de famille
- `Prénom.Individu` : prénom
- `Coordonnée.Coordonnée` : information de contact

La colonne `Coordonnée.Coordonnée` contient des valeurs hétérogènes :
- adresses e-mail professionnelles,
- mentions textuelles (ex. : *PAS D’EMAILING*, *DESINSCRIPTION E-MAILING*),
- valeurs de type *NC* (non communiqué),
- parfois des numéros de téléphone.

Ces éléments indiquent que ce champ nécessitera un traitement spécifique lors du nettoyage des données.

Feuille 2 — Taxe

Cette feuille contient une liste d’adresses e-mail correspondant aux verseurs de taxe 2025.

- Structure : une seule colonne
- Volumétrie observée/visible : 101 lignes
- Contenu : adresses e-mail uniquement

Feuille 3 — Pour e-mailing

Cette feuille contient une base d’adresses e-mail destinée à des actions d’e-mailing.

- Structure : une seule colonne (`Coordonnee.Coordonnee`)
- Volumétrie observée/visible : 5 563 lignes
- Contenu : adresses e-mail uniquement


## 2. Imports et chargement des dépendances
objectif : regrouper les bibliothèques Python nécessaires à l’analyse exploratoire des données.

In [1]:
import pandas as pd
import numpy as np


## 3. Chargement du fichier Excel
Objectif : charger le fichier Excel et vérifier l'accès aux différentes feuilles.

In [2]:
# chargement du fichier
file_path = "../data/raw/CONTACTS DIRECTEUR-DRH-SERVICE RH-TUTEURS 15-01-2026.xls"

xls = pd.ExcelFile(file_path)


## 4. Chargement de la feuille 1 — CONTACTS DIRECTEUR-DRH-SERVICE
Objectif : charger la première feuille et vérifier sa structure (dimensions, colonnes).


In [3]:
# afficher les noms des feuilles
xls.sheet_names


['CONTACTS DIRECTEUR-DRH-SERVICE ', 'Taxe', 'Pour e-mailing']

In [4]:
# charger la feuille "CONTACTS DIRECTEUR-DRH-SERVICE "  dans un DataFrame
df_contacts = pd.read_excel(
    "../data/raw/CONTACTS DIRECTEUR-DRH-SERVICE RH-TUTEURS 15-01-2026.xls",
    sheet_name="CONTACTS DIRECTEUR-DRH-SERVICE "
)


In [5]:
# afficher la taille du DataFrame
df_contacts.shape


(6799, 23)

In [6]:
# afficher les colonnes du DataFrame
df_contacts.columns


Index(['Code.Entité', 'Libellé.Entité', 'Code.Type d'entité',
       'Assujetti.Entité', 'Rue (ligne 1).Adresse', 'Rue (ligne 2).Adresse',
       'Rue (ligne 3).Adresse', 'Rue (ligne 4).Adresse', 'Code postal.Ville',
       'Nom.Ville', 'Code.Type d'adresse',
       'Nombre de stage (sans contrat pro)', 'Noms des stagiaires',
       'Nombre de contrat pro', 'Noms des alternants contrats pro',
       'Nombre apprentissage', 'Noms des apprentis', 'Code.Type d'événement',
       'Montant global.Taxe versement', 'Libellé.Titre', 'Nom.Individu',
       'Prénom.Individu', 'Coordonnée.Coordonnée'],
      dtype='str')

Observations :
- La feuille contient plus d’informations que prévu initialement, avec des dimensions “formation / taxe / événements” mélangées aux contacts :
    - je trouve bien les colonnes entité / adresse / contact → visibles initialement
    - je trouve aussi les colonnes stage / alternance / taxe / événement → non visibles initialement

In [7]:
# afficher les types de données des colonnes
df_contacts.dtypes


Code.Entité                               str
Libellé.Entité                            str
Code.Type d'entité                        str
Assujetti.Entité                      float64
Rue (ligne 1).Adresse                     str
Rue (ligne 2).Adresse                  object
Rue (ligne 3).Adresse                     str
Rue (ligne 4).Adresse                     str
Code postal.Ville                      object
Nom.Ville                                 str
Code.Type d'adresse                       str
Nombre de stage (sans contrat pro)    float64
Noms des stagiaires                       str
Nombre de contrat pro                 float64
Noms des alternants contrats pro          str
Nombre apprentissage                  float64
Noms des apprentis                        str
Code.Type d'événement                     str
Montant global.Taxe versement         float64
Libellé.Titre                             str
Nom.Individu                           object
Prénom.Individu                   

Observations :
  - les colonnes de comptage ('Nombre de stage', Nombre de contrat pro', etc) sont en float 64 -> probable présence de NaN
  - 'Assujetti.Entité est en float 64 -> booléen 'déguisé' (1/NaN au lieu de 1/0) -> quand c'est false, c'est remplacé par NaN
  - 'Code postal.Ville' en object -> probablement à cause des codes postaux avec zéros, CEDEX...
  - cette feuille mélange plusieurs dimensions métier dans une même table : des données de contact / des données pédagogiques liées à des dispositifs (stage, contrats pro, alternance, etc.) / des données administratives (taxe).

## 5. Valeurs manquantes

Objectif : mesurer la qualité des données par colonne afin d’identifier les champs critiques.


In [8]:
# afficher le nombre de valeurs manquantes par colonne
missing_count = df_contacts.isna().sum()
missing_count


Code.Entité                              0
Libellé.Entité                           0
Code.Type d'entité                      49
Assujetti.Entité                       331
Rue (ligne 1).Adresse                    2
Rue (ligne 2).Adresse                 4253
Rue (ligne 3).Adresse                 6488
Rue (ligne 4).Adresse                 6790
Code postal.Ville                        0
Nom.Ville                                0
Code.Type d'adresse                      0
Nombre de stage (sans contrat pro)    4368
Noms des stagiaires                   4368
Nombre de contrat pro                 4465
Noms des alternants contrats pro      4465
Nombre apprentissage                  5161
Noms des apprentis                    5161
Code.Type d'événement                 5785
Montant global.Taxe versement         5785
Libellé.Titre                           10
Nom.Individu                             0
Prénom.Individu                          1
Coordonnée.Coordonnée                  307
dtype: int6

In [9]:
# afficher le pourcentage de valeurs manquantes par colonne
missing_percent = (df_contacts.isna().mean() * 100).round(2)
missing_percent


Code.Entité                            0.00
Libellé.Entité                         0.00
Code.Type d'entité                     0.72
Assujetti.Entité                       4.87
Rue (ligne 1).Adresse                  0.03
Rue (ligne 2).Adresse                 62.55
Rue (ligne 3).Adresse                 95.43
Rue (ligne 4).Adresse                 99.87
Code postal.Ville                      0.00
Nom.Ville                              0.00
Code.Type d'adresse                    0.00
Nombre de stage (sans contrat pro)    64.24
Noms des stagiaires                   64.24
Nombre de contrat pro                 65.67
Noms des alternants contrats pro      65.67
Nombre apprentissage                  75.91
Noms des apprentis                    75.91
Code.Type d'événement                 85.09
Montant global.Taxe versement         85.09
Libellé.Titre                          0.15
Nom.Individu                           0.00
Prénom.Individu                        0.01
Coordonnée.Coordonnée           

In [10]:
# créer un DataFrame récapitulatif des valeurs manquantes
missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
}).sort_values("missing_percent", ascending=False)

missing_summary


,missing_count,missing_percent
Rue (ligne 4).Adresse,6790,99.87
Rue (ligne 3).Adresse,6488,95.43
Montant global.Taxe versement,5785,85.09
Code.Type d'événement,5785,85.09
Noms des apprentis,5161,75.91
Nombre apprentissage,5161,75.91
Noms des alternants contrats pro,4465,65.67
Nombre de contrat pro,4465,65.67
Noms des stagiaires,4368,64.24
Nombre de stage (sans contrat pro),4368,64.24


### Lecture des valeurs manquantes

L’analyse met en évidence plusieurs niveaux de complétude :

- Certaines colonnes d’adresse (lignes 3 et 4) sont quasiment vides (> 95 % de valeurs manquantes), ce qui limite fortement leur exploitabilité.
- Les colonnes liées à la formation (stages, contrats pro, apprentissages) et à la taxe d'apprentissage présentent des taux de valeurs manquantes élevés, cohérents avec le fait que ces informations ne concernent qu’une partie des entités.
- Les colonnes essentielles à l'identification des entités et des individus (identité, localisation, nom/prénom) sont très majoritairement complètes.
- La colonne `Coordonnée.Coordonnée` joue un rôle central pour les usages CRM. Elle regroupe plusieurs types d’informations : adresses e-mail, numéros de téléphone, ainsi que des valeurs textuelles indiquant le statut de contact (absence d’e-mail, désinscription, refus d’e-mailing, NC, etc.). Cette hétérogénéité met en évidence la nécessité, lors de la phase ETL, de distinguer les coordonnées par type (e-mail, téléphone) et de conserver explicitement l’information liée à l’autorisation ou non de l’e-mailing afin de permettre des usages CRM adaptés.

## 6. Analyse du contenu de la colonne Coordonnée.Coordonnée

Objectif : quantifier les différents types d’informations présents
(adresses e-mail, numéros de téléphone, statuts).


In [11]:
# normalisation minimale de la colonne Coordonnée.Coordonnée
# (conversion en texte et suppression des espaces) afin de permettre l’analyse
coord = df_contacts["Coordonnée.Coordonnée"].astype(str).str.strip()


In [12]:
# détection approximative des emails (présence du caractère '@')
mask_email = coord.str.contains(r"@", na=False)


In [13]:
# détection approximative des numéros de téléphone
# (présence d’une suite de chiffres)
mask_phone = coord.str.contains(r"\d{2}.*\d{2}", na=False)


In [14]:
# détection des valeurs ne correspondant ni à un email ni à un téléphone
mask_status = ~(mask_email | mask_phone)


In [15]:
# comptage du nombre de lignes correspondant à chaque type de coordonnée
email_count = mask_email.sum()
phone_count = mask_phone.sum()
status_count = mask_status.sum()

email_count, phone_count, status_count


(np.int64(6477), np.int64(21), np.int64(314))

In [16]:
# création d’un tableau de synthèse présentant
# le volume et la proportion de chaque type de coordonnée
coord_summary = pd.DataFrame({
    "type": ["email", "téléphone", "statut / autre"],
    "count": [email_count, phone_count, status_count],
    "percent": [
        round(email_count / len(df_contacts) * 100, 2),
        round(phone_count / len(df_contacts) * 100, 2),
        round(status_count / len(df_contacts) * 100, 2),
    ]
})

coord_summary


,type,count,percent
0,email,6477,95.26
1,téléphone,21,0.31
2,statut / autre,314,4.62


### Analyse du contenu de la colonne Coordonnée.Coordonnée

La colonne `Coordonnée.Coordonnée` jour un rôle central pour les usages CRM. Elle regroupe plusieurs tupes d'informations : environ 95% des lignes sont adresses e-mail, une très faible proportion correspond à des numéros de téléphone, tandis qu’une part non négligeable des valeurs correspond à des informations de statut (absence d’e-mail, refus ou désinscription à l’e-mailing, NC, etc.).

Ces résultats confirment l’intérêt de :
- distinguer les coordonnées par type (e-mail, téléphone),
- conserver explicitement l’information relative à l’autorisation ou non de l’e-mailing, afin de permettre des usages CRM conformes et adaptés.


## 7. Analyse des doublons email

Objectif : identifier la présence éventuelle de doublons d'adresses email, afin d'évaluer la qualité de la base pour un usage CRM/emailing.

In [17]:
# préparation de la colonne email :
# extraction de la colonne Coordonnée.Coordonnée
# et normalisation minimale pour l’analyse des doublons
emails = (
    df_contacts["Coordonnée.Coordonnée"]
    .astype(str)          # conversion en texte pour éviter les erreurs liées aux NaN
    .str.strip()          # suppression des espaces en début / fin
    .str.lower()          # mise en minuscules pour éviter les faux doublons
)


In [18]:
# sélection des lignes contenant une adresse e-mail (pour les isoler)
emails_only = emails[emails.str.contains("@", na=False)]


In [19]:
# détection des emails apparaissant plus d'une fois (doublons)
email_duplicates = emails_only[emails_only.duplicated(keep=False)]


In [20]:
# Quantification des doublons email :

# nombre total d'emails
total_emails = emails_only.shape[0]

# nombre d'emails distincts
unique_emails = emails_only.nunique()

# nombre d'emails dupliqués (en volume)
duplicate_emails_count = email_duplicates.shape[0]

total_emails, unique_emails, duplicate_emails_count


(6477, 5788, 1139)

In [21]:
# calcul du taux de doublons email
duplicate_rate = round((duplicate_emails_count / total_emails) * 100, 2)
duplicate_rate


17.59

In [ ]:
# comptage des emails les plus fréquemment présents
email_duplicate_summary = email_duplicates.value_counts().head(10)
email_duplicate_summary


Coordonnée.Coordonnée
thales.alternance-stage@pontoonsolutions.com    18
admin-stg-alt.navalgroup@manpowergroup.fr       13
stage-alternance@arkea.com                      13
stagesalternances.obssa@orange.com              12
admin-stg-alt.navalgroup@tapfin.fr              12
valerie.sable@capgemini.com                     11
eidprhecoles@e-i.com                            10
welcome.earlycareers@airbus.com                  9
sandra.belliure@capgemini.com                    8
formation.bretagne@ifremer.fr                    8
Name: count, dtype: int64

### Analyse des doublons d’adresses e-mail

L’analyse des adresses e-mail montre la présence de doublons dans la base.
Ces doublons peuvent correspondre :
- à des adresses génériques partagées (ex. contact@, info@),
- à des contacts communs à plusieurs entités ou services.

Ce point devra être pris en compte lors de la phase ETL, afin de définir les règles de gestion des contacts (unicité, rattachement aux entités, priorisation des coordonnées).

L’analyse des doublons montre que ceux-ci sont majoritairement associés aux contextes de stage et d’alternance. Cela suggère l’utilisation d’adresses e-mail communes ou génériques pour le suivi administratif de plusieurs stagiaires ou alternants.

Ces doublons ne traduisent donc pas nécessairement une mauvaise qualité des données, mais reflètent des pratiques métier spécifiques. Ce point devra être pris en compte lors de la définition des règles d’unicité et de rattachement des contacts dans le CRM.



## 8. Qualité des adresses e-mail

Objectif : évaluer la qualité des adresses e-mail afin d’anticiper les règles de validation et de nettoyage lors de la phase ETL.
